In [ ]:
import ee
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project=os.environ.get('EE_PROJECT'))
# Cloud authentication.
from google.colab import auth
auth.authenticate_user()

import ee
import os
import sys
import copy
import geemap.core as geemap
# Initialize a map object.


In [ ]:
# =================== PARÂMETROS ===================
start = '1985-01-01'
end = '2024-12-01'
# Usar a coleção completa
allGeometries = ee.FeatureCollection('projects/ee-rnvuefsppgm/assets/CAP_JOIN2')
allGeometries = allGeometries.map(lambda feat: feat.set('id_cod', 1))
print(allGeometries.getInfo(), "allGeometries")

region = allGeometries.geometry()

# Para visualização no mapa
import folium
my_map = folium.Map(location=[-15.78, -47.93], zoom_start=5)
my_map.add_child(folium.GeoJson(region.getInfo()))
my_map

assetPrefix = 'projects/rnvuefsppgm/assets/RSEI_CITIES'
exportScale = 30
exportCRS = None

In [ ]:
def maskLandsatC2L2(img):
    qa = img.select('QA_PIXEL')
    notDilated = qa.bitwiseAnd(1 << 1).eq(0)
    notCloud = qa.bitwiseAnd(1 << 3).eq(0)
    notShadow = qa.bitwiseAnd(1 << 4).eq(0)
    notSnow = qa.bitwiseAnd(1 << 5).eq(0)
    isClear = qa.bitwiseAnd(1 << 6).neq(0)

    hasRadSat = img.bandNames().contains('QA_RADSAT')
    notSaturated = ee.Image(ee.Algorithms.If(
        hasRadSat, img.select('QA_RADSAT').eq(0), ee.Image(1)
    ))

    # Correção para utilizar ee.Image.And() corretamente
    mask = ee.Image(notDilated).And(notCloud).And(notShadow).And(notSnow).And(isClear).And(notSaturated)
    return img.updateMask(mask)



In [ ]:
def harmonizeLandsatC2L2_L89(img):
    img = maskLandsatC2L2(img)

    hasB10 = img.bandNames().contains('ST_B10')  # L8/L9
    oliSR = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
    outSR = ['blue','green','red','nir','swir1','swir2']

    # SR dos sensores harmonizada
    srSelected = img.select(oliSR).multiply(0.0000275).add(-0.2).rename(outSR)
    srSelected = ee.Image(srSelected)

    # ST (Kelvin) a partir do produto de temperatura de superfície do C2 L2
    stKelvin = img.select('ST_B10').multiply(0.00341802).add(149.0).rename('st')
    srSelected = srSelected.addBands(stKelvin).toFloat()
    srSelected = addIndices(srSelected)

    return srSelected.copyProperties(img, img.propertyNames())



In [ ]:
def harmonizeLandsatC2L2_l57(img):
    img = maskLandsatC2L2(img)
    hasB06 = img.bandNames().contains('ST_B6')  # L5/L7

    tmSR = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
    outSR = ['blue','green','red','nir','swir1','swir2']

    # SR dos sensores harmonizada
    srSelected = img.select(tmSR).multiply(0.0000275).add(-0.2).rename(outSR)

    # ST (Kelvin) a partir do produto de temperatura de superfície do C2 L2
    stKelvin = img.select('ST_B6').multiply(0.00341802).add(149.0).rename('st')

    srSelected = srSelected.addBands(stKelvin).toFloat()
    srSelected = addIndices(srSelected)

    return srSelected.copyProperties(img, img.propertyNames())



In [ ]:
# =================== ÍNDICES (NDBSI + NDVI + LST + Wetness) ===================
def addIndices(img):
    # NDBSI
    ndbsi = img.expression(
        '((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue))',
        {
            'swir1': img.select('swir1'),
            'red': img.select('red'),
            'nir': img.select('nir'),
            'blue': img.select('blue')
        }
    ).rename('NDBSI')

    # NDVI
    ndvi = img.normalizedDifference(['nir', 'red']).rename('NDVI')

    # LST em °C
    lstC = img.select('st').subtract(273.15).rename('LST')

    # Tasseled Cap (Wetness)
    X = img.select(['blue', 'green', 'red', 'nir', 'swir1', 'swir2']).toArray().toArray(1)

    TCT_TM = ee.Array([
        [0.3037, 0.2793, 0.4743, 0.5585, 0.5082, 0.1863],  # Brightness (TCB)
        [-0.2848, -0.2435, -0.5436, 0.7243, 0.0840, -0.1800],  # Greenness (TCG)
        [0.1509, 0.1973, 0.3279, 0.3406, -0.7112, -0.4572]   # Wetness (TCW)
    ])

    TCT_OLI = ee.Array([
        [0.3029, 0.2786, 0.4733, 0.5599, 0.5080, 0.1872],
        [-0.2941, -0.2430, -0.5424, 0.7276, 0.0713, -0.1608],
        [0.1511, 0.1973, 0.3283, 0.3407, -0.7117, -0.4559]
    ])

    hasB10 = img.bandNames().contains('ST_B10')  # OLI (L8/L9)
    A = ee.Array(ee.Algorithms.If(hasB10, TCT_OLI, TCT_TM))

    tcap = ee.Image(A).matrixMultiply(X).arrayProject([0]).arrayFlatten([['TCB', 'TCG', 'TCW']])

    # Banda "wetness" explicitamente exposta
    wet = tcap.select('TCW').rename('wetness')

    return img.addBands([ndbsi, ndvi, lstC, tcap, wet])



In [ ]:
# Processar os dados
lst_codSHP = allGeometries.aggregate_array('CD_MUN')

# Agora usamos um loop for no lugar de 'forEach'
lst_codSHP_info = lst_codSHP.getInfo()  # Obtém a lista de códigos de municípios

for codMun in lst_codSHP_info:
    export_images(codMun)  # Chama a função de exportação para cada código de município

def export_images(codMun):
    featCol = allGeometries.filter(ee.Filter.eq('CD_MUN', codMun))
    feat = featCol.first()
    mask_feat = featCol.reduceToImage(['id_cod'], ee.Reducer.first()).gt(0)

    landsatAll_regL5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2') \
        .filterDate('1985-01-01', '2012-01-01') \
        .filterBounds(feat.geometry()) \
        .map(harmonizeLandsatC2L2_l57)

    lst_idcod = landsatAll_regL5.aggregate_array('system:index')

    def process_images(idSystem):
        img_temp = ee.Image(landsatAll_regL5.filter(ee.Filter.eq('system:index', idSystem)).first())
        img_temp = img_temp.updateMask(mask_feat) \
            .copyProperties(feat, ['NM_MUN', 'CD_MUN']) \
            .set('system:footprint', feat.geometry())

        param_exp = {
            'image': img_temp,
            'description': idSystem,
            'assetId': assetPrefix + '/' + idSystem,
            'region': feat.geometry(),
            'scale': exportScale,
            'maxPixels': 1e13,
            'crs': exportCRS
        }
        ee.batch.Export.image.toAsset(param_exp).start()

    lst_idcod.getInfo().forEach(process_images)

def maskLandsatC2L2(img):
    qa = img.select('QA_PIXEL')
    notDilated = qa.bitwiseAnd(1 << 1).eq(0)
    notCloud = qa.bitwiseAnd(1 << 3).eq(0)
    notShadow = qa.bitwiseAnd(1 << 4).eq(0)
    notSnow = qa.bitwiseAnd(1 << 5).eq(0)
    isClear = qa.bitwiseAnd(1 << 6).neq(0)

    hasRadSat = img.bandNames().contains('QA_RADSAT')
    notSaturated = ee.Image(ee.Algorithms.If(
        hasRadSat, img.select('QA_RADSAT').eq(0), ee.Image(1)
    ))

    # Correção: substituir '.and()' por 'ee.Image.and()'
    mask = ee.Image(notDilated).And(notCloud).And(notShadow).And(notSnow).And(isClear).And(notSaturated)
    return img.updateMask(mask)
